# Stage 3 — Results Visualization

Loads your saved SNN checkpoint and the real, 100%-real test set, and produces:
1. ROC curve (with AUC)
2. Precision-Recall curve
3. Confusion matrix heatmap
4. Bloom probability map — actual labels vs. predicted probability, plotted at real lat/lon

All plots use only the held-out real test set — nothing synthetic touches these numbers.


In [ ]:
from google.colab import files
import os

# Upload files from your laptop
uploaded = files.upload()

# Set paths
MODEL_PATH = '/content/snn_bloom_classifier_tuned.pt'
TEST_CSV = '/content/algae_dataset_test.csv'
OUTPUT_DIR = '/content/plots'

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify
print("\nChecking files...")

print("Model:", os.path.exists(MODEL_PATH))
print("Test CSV:", os.path.exists(TEST_CSV))
print("Output directory:", os.path.exists(OUTPUT_DIR))

## 1. Config

In [ ]:
MODEL_PATH = '/content/snn_bloom_classifier_tuned.pt'
TEST_CSV   = '/content/algae_dataset_test.csv'
OUTPUT_DIR = '/content/plots'

## 2. Install & imports

In [ ]:
!pip install snntorch scikit-learn matplotlib -q

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import snntorch as snn
from snntorch import surrogate
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, average_precision_score,
                              confusion_matrix, accuracy_score, f1_score)

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Rebuild the model and load the saved weights

In [ ]:
class SNNClassifier(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs, beta):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid()
        self.fc1  = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2  = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x, num_steps):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        spk2_rec = []
        for _ in range(num_steps):
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_rec.append(spk2)
        return torch.stack(spk2_rec, dim=0)


checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
cfg = checkpoint['config']
selected_features = checkpoint['selected_features']
scaler_mean = np.array(checkpoint['scaler_mean'])
scaler_scale = np.array(checkpoint['scaler_scale'])

model = SNNClassifier(cfg['num_inputs'], cfg['num_hidden'], cfg['num_outputs'], cfg['beta'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('Loaded model. Selected features:', selected_features)
print('Num steps:', cfg['num_steps'])

## 4. Load the real test set and reproduce the exact preprocessing
Scales with the *same* mean/scale the model was trained with (saved in the checkpoint), then
selects the same feature subset BCPO chose — reproducing training-time preprocessing exactly.

In [ ]:
test_df = pd.read_csv(TEST_CSV)

non_feature_cols = ['longitude', 'latitude', 'cyan_dn', 'label', 'source_date', 'is_synthetic']
feature_columns = [c for c in test_df.columns if c not in non_feature_cols]
assert len(feature_columns) == 31, f'Expected 31 features, found {len(feature_columns)}'

X_test_all = test_df[feature_columns].values.astype(np.float64)
X_test_scaled = (X_test_all - scaler_mean) / scaler_scale

selected_idx = [feature_columns.index(f) for f in selected_features]
X_test_sel = X_test_scaled[:, selected_idx]

y_test = test_df['label'].values.astype(np.int64)
X_test_t = torch.tensor(X_test_sel, dtype=torch.float32)

print('Test set:', X_test_t.shape, ' Real, held-out dates:',
      sorted(test_df['source_date'].unique()) if 'source_date' in test_df.columns else 'n/a')

## 5. Run inference — predictions and probabilities

In [ ]:
with torch.no_grad():
    spk_rec = model(X_test_t, cfg['num_steps'])
    spk_counts = spk_rec.sum(dim=0)                       # [N, 2] -- used as logits
    probs = torch.softmax(spk_counts, dim=1).numpy()       # [N, 2] -- class probabilities
    preds = spk_counts.argmax(dim=1).numpy()

y_prob_bloom = probs[:, 1]   # probability of class 1 (bloom)

print('Accuracy:', round(accuracy_score(y_test, preds), 4))
print('F1:', round(f1_score(y_test, preds), 4))

## 6. ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob_bloom)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='#d62728', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve -- SNN Bloom Classifier (real test set)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curve.png'), dpi=150)
plt.show()

## 7. Precision-Recall curve

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_prob_bloom)
avg_precision = average_precision_score(y_test, y_prob_bloom)

plt.figure(figsize=(6, 6))
plt.plot(recall, precision, color='#1f77b4', lw=2, label=f'PR curve (AP = {avg_precision:.3f})')
plt.axhline(y=(y_test == 1).mean(), color='gray', lw=1, linestyle='--', label='No-skill baseline')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve -- SNN Bloom Classifier')
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'precision_recall_curve.png'), dpi=150)
plt.show()

## 8. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, preds)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['No Bloom', 'Bloom'])
ax.set_yticklabels(['No Bloom', 'Bloom'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix -- Real Test Set')

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)

plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 9. Bloom probability map
Left: actual labels at their real lat/lon. Right: predicted bloom probability at the same
locations. Visual agreement between the two panels is a strong, intuitive sanity check that the
model is picking up spatially coherent bloom structure, not noise.

In [ ]:
assert 'longitude' in test_df.columns and 'latitude' in test_df.columns, \
    'Test file has no coordinates -- re-export from Stage 1 with lon/lat retained.'

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

sc0 = axes[0].scatter(test_df['longitude'], test_df['latitude'],
                       c=y_test, cmap='RdYlGn_r', s=10, vmin=0, vmax=1)
axes[0].set_title('Actual Labels (real CyAN data)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
cb0 = plt.colorbar(sc0, ax=axes[0], ticks=[0, 1])
cb0.ax.set_yticklabels(['No Bloom', 'Bloom'])

sc1 = axes[1].scatter(test_df['longitude'], test_df['latitude'],
                       c=y_prob_bloom, cmap='RdYlGn_r', s=10, vmin=0, vmax=1)
axes[1].set_title('Predicted Bloom Probability (SNN)')
axes[1].set_xlabel('Longitude')
cb1 = plt.colorbar(sc1, ax=axes[1])
cb1.set_label('P(bloom)')

fig.suptitle('Lake Erie -- Real Test Set: Actual vs. Predicted', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'bloom_probability_map.png'), dpi=150)
plt.show()

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Confusion matrix
cm = np.array([
    [1784, 152],
    [86, 3318]
])

labels = ["No Bloom", "Bloom"]

# Professional colour palette
header_color = "#263B55"
correct_color = "#DCE5EF"
error_color = "#F1F3F5"
text_color = "#263238"

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axis("off")

# Table values
data = [
    ["Actual / Predicted", "No Bloom", "Bloom"],
    ["No Bloom", "1,784", "152"],
    ["Bloom", "86", "3,318"]
]

table = ax.table(
    cellText=data,
    cellLoc="center",
    loc="center",
    colWidths=[0.35, 0.33, 0.33]
)

table.auto_set_font_size(False)
table.set_fontsize(16)

# Style each cell
for (row, col), cell in table.get_celld().items():

    cell.set_height(0.25)
    cell.set_edgecolor("#D0D7DE")
    cell.set_linewidth(0.8)

    # Header
    if row == 0:
        cell.set_facecolor(header_color)
        cell.get_text().set_color("white")
        cell.get_text().set_weight("bold")

    # Row labels
    elif col == 0:
        cell.set_facecolor("#E9EDF2")
        cell.get_text().set_color(text_color)
        cell.get_text().set_weight("bold")

    # Correct predictions
    elif row == col:
        cell.set_facecolor(correct_color)
        cell.get_text().set_color(text_color)
        cell.get_text().set_weight("bold")

    # Incorrect predictions
    else:
        cell.set_facecolor(error_color)
        cell.get_text().set_color(text_color)

# Title
ax.set_title(
    "Confusion Matrix",
    fontsize=19,
    fontweight="bold",
    color=header_color,
    pad=22
)

plt.tight_layout()

# Save high-resolution image
plt.savefig(
    "confusion_matrix_professional.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


## Note on the training-curve gap
This notebook only plots the *evaluation* side (ROC/PR/confusion matrix/map) because Stage 2 didn't
persist per-epoch loss/accuracy history to disk — only the final trained weights were saved. If you
want a loss/accuracy-over-epochs plot too, add two lines to Stage 2's training loop to append
`train_losses`/`train_accs` to a list and `json.dump` them alongside the model next time you train,
and this notebook can be extended to plot them.
